## GAN and CNN Hybrid Project

This notebook will set up a Generative Adversarial Network (GAN) with Convolutional Neural Network (CNN) components for image generation.

In [ ]:
# Hyperparameters
LATENT_DIM = 100
EPOCHS = 50
BATCH_SIZE = 256
BUFFER_SIZE = 60000 # For shuffling the dataset


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
import numpy as np
import matplotlib.pyplot as plt

# Ensure TensorFlow is using a GPU if available
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

### 1. Define the Generator Model

The generator takes a random noise vector (`LATENT_DIM`) as input and upsamples it through a series of transposed convolution layers to produce an image. The output layer uses a `tanh` activation to ensure pixel values are in the range [-1, 1].

In [ ]:
# Assuming image shape based on common GAN setups, e.g., 28x28 for MNIST, 64x64 for more complex images.
# Since x_train is an ndarray, we need to infer its shape. Let's assume it's (num_samples, height, width, channels).
# For now, we'll use a placeholder `image_shape` which will need to be adjusted based on x_train's actual shape.
# Based on the typical GAN examples, let's assume an output image shape like (64, 64, 1) or (28, 28, 1).
# If x_train is available, we can get the shape directly from it.

# Placeholder for image_shape. User might need to adjust this.
# Assuming x_train is a batch of grayscale images like MNIST or fashion MNIST, scaled to [-1, 1].
# For demonstration, let's use a common target output shape for a basic GAN.
# If x_train is defined and has a shape like (N, H, W, C), then target_image_shape = x_train.shape[1:]
# For example, if x_train.shape is (60000, 28, 28, 1):
# target_image_shape = (28, 28, 1)

# For now, let's use a common GAN output shape for demonstration
target_image_height = 64
target_image_width = 64
target_image_channels = 1 # Grayscale

target_image_shape = (target_image_height, target_image_width, target_image_channels)

def make_generator_model():
    model = tf.keras.Sequential()
    model.add(layers.Dense(4*4*256, use_bias=False, input_shape=(LATENT_DIM,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Reshape((4, 4, 256)))
    assert model.output_shape == (None, 4, 4, 256) # Note: None is for the batch size

    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    assert model.output_shape == (None, 8, 8, 128)

    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    assert model.output_shape == (None, 16, 16, 64)

    model.add(layers.Conv2DTranspose(target_image_channels, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'))
    assert model.output_shape == (None, 32, 32, target_image_channels)

    # If target_image_shape is 64x64, need one more layer
    if target_image_height == 64 or target_image_width == 64:
        model.add(layers.Conv2DTranspose(target_image_channels, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'))
        assert model.output_shape == (None, 64, 64, target_image_channels)

    return model

generator = make_generator_model()

# Test the generator
noise = tf.random.normal([1, LATENT_DIM])
generated_image = generator(noise, training=False)
plt.imshow(generated_image[0, :, :, 0] * 0.5 + 0.5, cmap='gray')
plt.title("Initial Generated Image (before training)")
plt.axis('off')
plt.show()

### 2. Define the Discriminator Model

The discriminator is a CNN-based image classifier that takes an image as input and outputs a single scalar prediction indicating whether the image is 'real' (from the training data) or 'fake' (generated by the generator). It uses `LeakyReLU` activations and `Dropout` for stability.

In [ ]:
def make_discriminator_model():
    model = tf.keras.Sequential()
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=target_image_shape))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    return model

discriminator = make_discriminator_model()

# Test the discriminator with the generated image
decision = discriminator(generated_image)
print(f"Discriminator's initial decision for a fake image: {decision.numpy()}")

### 3. Define Loss Functions and Optimizers

We'll use `BinaryCrossentropy` for both the generator and discriminator loss. The optimizers (`Adam`) will have different learning rates to control their respective training paces.

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

### 4. Define the Training Step

The `train_step` function encapsulates one step of the GAN training process. It trains both the discriminator and the generator by computing their respective losses and applying gradients.

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, LATENT_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss

### 5. Prepare the Dataset

We need to process the `x_train` data, assuming it contains images. This involves batching, shuffling, and potentially reshaping/normalizing the data if not already done.

In [ ]:
# Assuming x_train is already loaded and preprocessed to [-1, 1] range and correct shape.
# If x_train needs reshaping or normalization, add it here.
# For example, if x_train is (num_samples, 28, 28) for grayscale images:
# x_train = x_train.reshape(x_train.shape[0], 28, 28, 1).astype('float32')
# x_train = (x_train - 127.5) / 127.5 # Normalize to [-1, 1]

# Check the shape of x_train and adapt target_image_shape if necessary
if 'x_train' in globals() and isinstance(x_train, np.ndarray):
    print(f"x_train shape: {x_train.shape}")
    # Assuming x_train is (num_samples, height, width, channels)
    if x_train.ndim == 4:
        target_image_shape = x_train.shape[1:]
        print(f"Adjusted target_image_shape to: {target_image_shape}")
    elif x_train.ndim == 3: # (num_samples, height, width)
        target_image_shape = x_train.shape[1:] + (1,)
        print(f"Adjusted target_image_shape to: {target_image_shape} (assuming grayscale)")
        x_train = np.expand_dims(x_train, axis=-1)
    else:
        print("Warning: x_train has an unexpected number of dimensions. Please check your data.")


train_dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

### 6. Training Loop

The main training loop iterates for `EPOCHS`. In each epoch, it goes through the `train_dataset` and calls the `train_step` function. It also generates and saves example images periodically to visualize the generator's progress.

In [ ]:
def generate_and_save_images(model, epoch, test_input):
    predictions = model(test_input, training=False)

    fig = plt.figure(figsize=(4, 4))

    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i+1)
        plt.imshow(predictions[i, :, :, 0] * 0.5 + 0.5, cmap='gray')
        plt.axis('off')

    plt.savefig('image_at_epoch_{:04d}.png'.format(epoch))
    plt.show()

# We'll reuse this seed overtime (so it's easier to visualize progress)
seed = tf.random.normal([16, LATENT_DIM])

def train(dataset, epochs):
    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")
        total_gen_loss = 0.0
        total_disc_loss = 0.0
        num_batches = 0
        for image_batch in dataset:
            gen_loss, disc_loss = train_step(image_batch)
            total_gen_loss += gen_loss
            total_disc_loss += disc_loss
            num_batches += 1

        avg_gen_loss = total_gen_loss / num_batches
        avg_disc_loss = total_disc_loss / num_batches
        print(f"  Generator Loss: {avg_gen_loss:.4f}, Discriminator Loss: {avg_disc_loss:.4f}")

        # Produce images for the GIF as we go
        if (epoch + 1) % 5 == 0 or epoch == 0:
            generate_and_save_images(generator, epoch + 1, seed)

    # Generate after the final epoch
    generate_and_save_images(generator, epochs, seed)

train(train_dataset, EPOCHS)
print("\nTraining complete!")

### 7. Save the Trained Generator Model

After training, it's good practice to save the generator model's weights so you can reuse it later without retraining. You can save the entire model using `model.save()`.

In [ ]:
import os

# Create a directory to save models if it doesn't exist
model_save_dir = './saved_models'
os.makedirs(model_save_dir, exist_ok=True)

generator_save_path = os.path.join(model_save_dir, 'generator_model.h5')
generator.save(generator_save_path)

print(f"Generator model saved to: {generator_save_path}")